# Baseline Speculative Decoding

**COMS 4232 Final Project — Nikhil Sharma (ns3942) & Mertay Dayanc (md4437)**

This notebook implements the standard speculative decoding algorithm (Algorithm 1) from
[Chen et al. (2023)](https://arxiv.org/abs/2302.01318) / [Leviathan et al. (2023)](https://arxiv.org/abs/2211.17192).

**Setup:**
- **Draft model ($p$):** GPT-2 small (124M parameters)
- **Target model ($q$):** GPT-2 large (774M parameters)
- **Shared tokenizer:** GPT-2 tokenizer

**Algorithm (per verification round):**

For each drafted token $\tilde{x}_t$ at positions $t = n, \dots, n + K - 1$:
1. Sample $r \sim \text{Uniform}[0, 1]$.
2. If $r \leq \min\!\Big(1,\; \frac{q_t(\tilde{x}_t)}{p_t(\tilde{x}_t)}\Big)$, **accept** $x_n = \tilde{x}_t$ and advance $n$.
3. Otherwise, **reject**: sample $x_n \sim [q_t - p_t]_+(\cdot)$ (the normalized residual), advance $n$, and break.
4. If all $K$ tokens are accepted, sample one **bonus token** from $q_{n+K}(\cdot)$.

## 1. Imports & Model Loading

In [7]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

# Use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load shared tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Load draft model (GPT-2 small, 124M)
draft_model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
draft_model.eval()

# Load target model (GPT-2 large, 774M)
# target_model = AutoModelForCausalLM.from_pretrained("gpt2-large").to(device)

# Load target model (GPT-2 XL, 1.5B)
target_model = AutoModelForCausalLM.from_pretrained("gpt2-xl").to(device)
target_model.eval()

print(f"Draft model parameters:  {sum(p.numel() for p in draft_model.parameters()):,}")
print(f"Target model parameters: {sum(p.numel() for p in target_model.parameters()):,}")
print(f"Vocabulary size: {tokenizer.vocab_size}")

Using device: cuda


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Draft model parameters:  124,439,808
Target model parameters: 1,557,611,200
Vocabulary size: 50257


## 2. Draft Function

The draft model runs autoregressively for K steps. At each step we:
1. Run a forward pass to get logits at the current position.
2. Convert logits to a probability distribution via softmax.
3. Sample a token from that distribution.
4. Append the token and save the probability distribution.

Returns the K drafted tokens and their corresponding draft probability distributions (needed for the acceptance/rejection step).

In [8]:
@torch.no_grad()
def draft(input_ids: torch.Tensor, K: int) -> tuple[list[int], list[torch.Tensor]]:
    """
    Run the draft model autoregressively for K steps.

    Args:
        input_ids: Token IDs of the current prefix, shape (1, seq_len).
        K: Number of draft tokens to generate.

    Returns:
        draft_tokens: List of K sampled token IDs.
        draft_probs:  List of K probability distributions (each shape (vocab_size,))
                      — the draft model's distribution at each step.
    """
    draft_tokens = []
    draft_probs = []
    current_ids = input_ids.clone()

    for _ in range(K):
        # Forward pass through draft model
        outputs = draft_model(current_ids)
        # Logits at the last position -> shape (vocab_size,)
        logits = outputs.logits[0, -1, :]
        # Convert to probabilities
        probs = F.softmax(logits, dim=-1)
        # Sample a token
        token = torch.multinomial(probs, num_samples=1).item()

        draft_tokens.append(token)
        draft_probs.append(probs)

        # Append the sampled token for the next step
        current_ids = torch.cat(
            [current_ids, torch.tensor([[token]], device=device)], dim=1
        )

    return draft_tokens, draft_probs

## 3. Verification Function (Accept / Reject)

Implements Algorithm 1 from the paper. The target model verifies the drafted tokens in **one forward pass**.

**Notation (matching the paper):**
- $p_t$ = draft model distribution at step $t$
- $q_t$ = target model distribution at step $t$
- $\tilde{x}_t$ = drafted token at step $t$

**Key indexing detail:** If the prefix has length `n`, then `logits[0, n-1+t, :]` gives the target model's distribution that predicts the token at position `n+t` (the `t`-th drafted token).

**Accept/reject loop (lines 3–12 of Algorithm 1):**
- Accept if $r \leq \min(1,\; q_t(\tilde{x}_t) / p_t(\tilde{x}_t))$
- On rejection, sample from $[q_t - p_t]_+(\cdot)$
- If all K accepted, sample a bonus token from $q_{n+K}$

In [9]:
@torch.no_grad()
def verify(
    prefix_ids: torch.Tensor,
    draft_tokens: list[int],
    draft_probs: list[torch.Tensor],
) -> tuple[list[int], int]:
    """
    Verify drafted tokens using the target model (Algorithm 1, lines 3-12).

    Notation (matching the paper):
        p_t = draft model distribution at step t
        q_t = target model distribution at step t
        x_t = drafted token at step t

    Args:
        prefix_ids:   Token IDs of the prefix *before* drafting, shape (1, n).
        draft_tokens: List of K drafted token IDs.
        draft_probs:  List of K draft distributions p_t, each shape (vocab_size,).

    Returns:
        accepted_tokens: List of accepted (and possibly resampled) tokens.
        n_accepted:      Number of draft tokens accepted (0 to K).
    """
    K = len(draft_tokens)
    n = prefix_ids.shape[1]  # prefix length

    # Build the full sequence: prefix + draft tokens
    draft_tensor = torch.tensor([draft_tokens], device=device)
    full_ids = torch.cat([prefix_ids, draft_tensor], dim=1)  # shape (1, n + K)

    # Single forward pass through the target model
    outputs = target_model(full_ids)
    all_logits = outputs.logits  # shape (1, n + K, vocab_size)

    accepted_tokens = []
    n_accepted = 0

    # Lines 3-12: for t = n : n+K-1
    for t in range(K):
        # q_t: target model's distribution for position (n + t)
        # Logits at index (n - 1 + t) predict the token at position (n + t)
        q_t = F.softmax(all_logits[0, n - 1 + t, :], dim=-1)

        # p_t: draft model's distribution at this step
        p_t = draft_probs[t]

        # x_t: the drafted token
        x_t = draft_tokens[t]

        # Line 4: sample r ~ Uniform[0, 1]
        r = torch.rand(1).item()

        # Line 5: if r <= min(1, q_t(x_t) / p_t(x_t))
        q_x = q_t[x_t].item()
        p_x = p_t[x_t].item()

        if p_x == 0:
            acceptance_prob = 0.0
        else:
            acceptance_prob = min(1.0, q_x / p_x)

        if r <= acceptance_prob:
            # Line 6: accept x_n = x_t, n <- n + 1
            accepted_tokens.append(x_t)
            n_accepted += 1
        else:
            # Line 8: sample x_n ~ [q_t - p_t]_+(.)
            residual = torch.clamp(q_t - p_t, min=0.0)
            residual_sum = residual.sum()
            if residual_sum > 0:
                residual = residual / residual_sum
            else:
                # Fallback (shouldn't happen in theory)
                residual = q_t
            resampled_token = torch.multinomial(residual, num_samples=1).item()
            accepted_tokens.append(resampled_token)
            # Line 9: n <- n + 1. Break.
            return accepted_tokens, n_accepted

    # All K tokens accepted — sample a bonus token from q_{n+K}
    bonus_probs = F.softmax(all_logits[0, n + K - 1, :], dim=-1)
    bonus_token = torch.multinomial(bonus_probs, num_samples=1).item()
    accepted_tokens.append(bonus_token)
    n_accepted = K

    return accepted_tokens, n_accepted

## 4. Speculative Decoding — Outer Loop

The outer loop repeatedly:
1. Calls `draft()` to get K candidate tokens from the draft model.
2. Calls `verify()` to check them against the target model.
3. Appends the accepted tokens to the running sequence.
4. Repeats until we hit `max_new_tokens` or generate the EOS token.

We also track per-round acceptance counts to compute the overall acceptance rate.

In [10]:
@torch.no_grad()
def speculative_decode(
    prompt: str,
    K: int = 4,
    max_new_tokens: int = 50,
) -> dict:
    """
    Generate text using speculative decoding.

    Args:
        prompt:         The input text prompt.
        K:              Number of draft tokens per round.
        max_new_tokens: Maximum number of new tokens to generate.

    Returns:
        Dictionary with:
            - "text": generated text (str)
            - "tokens_generated": total tokens generated
            - "rounds": number of draft-verify rounds
            - "total_accepted": total draft tokens accepted
            - "total_drafted": total draft tokens proposed
            - "acceptance_rate": fraction of draft tokens accepted
            - "time": wall-clock time in seconds
            - "target_calls": number of target model forward passes
            - "draft_calls": number of draft model forward passes
    """
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated_ids = input_ids.clone()
    eos_token_id = tokenizer.eos_token_id

    total_accepted = 0
    total_drafted = 0
    rounds = 0
    target_calls = 0
    draft_calls = 0

    start_time = time.time()

    tokens_generated = 0
    while tokens_generated < max_new_tokens:
        # Step 1: Draft K tokens
        draft_tokens, draft_probs = draft(generated_ids, K)
        draft_calls += K  # one forward pass per draft step

        # Step 2: Verify
        accepted_tokens, n_accepted = verify(generated_ids, draft_tokens, draft_probs)
        target_calls += 1  # single forward pass for verification

        # Bookkeeping
        total_accepted += n_accepted
        total_drafted += K
        rounds += 1

        # Append accepted tokens to the sequence
        new_tokens = torch.tensor([accepted_tokens], device=device)
        generated_ids = torch.cat([generated_ids, new_tokens], dim=1)
        tokens_generated += len(accepted_tokens)

        # Check for EOS in accepted tokens
        if eos_token_id in accepted_tokens:
            break

    elapsed = time.time() - start_time

    # Decode the full output
    output_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    acceptance_rate = total_accepted / total_drafted if total_drafted > 0 else 0.0

    return {
        "text": output_text,
        "tokens_generated": tokens_generated,
        "rounds": rounds,
        "total_accepted": total_accepted,
        "total_drafted": total_drafted,
        "acceptance_rate": acceptance_rate,
        "time": elapsed,
        "target_calls": target_calls,
        "draft_calls": draft_calls,
    }

## 5. Vanilla Autoregressive Baseline

Standard autoregressive generation using the **target model only** — one forward pass per token.
This is what speculative decoding aims to match in output quality while reducing wall-clock time.

In [11]:
@torch.no_grad()
def autoregressive_generate(
    prompt: str,
    max_new_tokens: int = 50,
) -> dict:
    """
    Standard autoregressive generation using the target model.

    Args:
        prompt:         The input text prompt.
        max_new_tokens: Maximum number of new tokens to generate.

    Returns:
        Dictionary with:
            - "text": generated text
            - "tokens_generated": number of tokens generated
            - "time": wall-clock time in seconds
            - "target_calls": number of target model forward passes
    """
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated_ids = input_ids.clone()
    eos_token_id = tokenizer.eos_token_id

    start_time = time.time()

    tokens_generated = 0
    for _ in range(max_new_tokens):
        outputs = target_model(generated_ids)
        logits = outputs.logits[0, -1, :]
        probs = F.softmax(logits, dim=-1)
        token = torch.multinomial(probs, num_samples=1).item()

        generated_ids = torch.cat(
            [generated_ids, torch.tensor([[token]], device=device)], dim=1
        )
        tokens_generated += 1

        if token == eos_token_id:
            break

    elapsed = time.time() - start_time
    output_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    return {
        "text": output_text,
        "tokens_generated": tokens_generated,
        "time": elapsed,
        "target_calls": tokens_generated,
    }

## 6. Run Comparison

We run both methods on the same prompt and compare:
- **Acceptance rate**: What fraction of draft tokens did the target model accept?
- **Wall-clock time**: How fast is speculative decoding vs. vanilla autoregressive?
- **Target model calls**: The key metric — speculative decoding should use far fewer target forward passes.

Note: Since both methods sample stochastically, the outputs will differ across runs. The key property is that speculative decoding produces samples from the **same distribution** as the target model.

In [12]:
prompt = "The future of artificial intelligence is"
K = 4
max_new_tokens = 50

print(f"Prompt: \"{prompt}\"")
print(f"Draft length K = {K}, max_new_tokens = {max_new_tokens}")
print("=" * 70)

# --- Speculative Decoding ---
print("\n[Speculative Decoding]")
spec_result = speculative_decode(prompt, K=K, max_new_tokens=max_new_tokens)

print(f"Generated text:\n  {spec_result['text']}\n")
print(f"  Tokens generated:  {spec_result['tokens_generated']}")
print(f"  Rounds:            {spec_result['rounds']}")
print(f"  Draft tokens proposed: {spec_result['total_drafted']}")
print(f"  Draft tokens accepted: {spec_result['total_accepted']}")
print(f"  Acceptance rate:   {spec_result['acceptance_rate']:.2%}")
print(f"  Target model calls: {spec_result['target_calls']}")
print(f"  Draft model calls:  {spec_result['draft_calls']}")
print(f"  Wall-clock time:   {spec_result['time']:.2f}s")

# --- Vanilla Autoregressive ---
print("\n" + "=" * 70)
print("\n[Vanilla Autoregressive (target model only)]")
ar_result = autoregressive_generate(prompt, max_new_tokens=max_new_tokens)

print(f"Generated text:\n  {ar_result['text']}\n")
print(f"  Tokens generated:   {ar_result['tokens_generated']}")
print(f"  Target model calls: {ar_result['target_calls']}")
print(f"  Wall-clock time:    {ar_result['time']:.2f}s")

# --- Summary ---
print("\n" + "=" * 70)
print("\n[Summary]")
print(f"  Speculative decoding used {spec_result['target_calls']} target calls "
      f"vs {ar_result['target_calls']} for autoregressive.")
if ar_result['time'] > 0:
    speedup = ar_result['time'] / spec_result['time']
    print(f"  Wall-clock speedup: {speedup:.2f}x")
print(f"  Token acceptance rate: {spec_result['acceptance_rate']:.2%}")

Prompt: "The future of artificial intelligence is"
Draft length K = 4, max_new_tokens = 50

[Speculative Decoding]
Generated text:
  The future of artificial intelligence is a real concern right now. Google could make an incredible amount of money answering your queries, then derail the promises of online advertising with late price cuts and sloppy data. Why should people think about the future of these engines when the profit motive is, on the whole

  Tokens generated:  52
  Rounds:            18
  Draft tokens proposed: 72
  Draft tokens accepted: 34
  Acceptance rate:   47.22%
  Target model calls: 18
  Draft model calls:  72
  Wall-clock time:   1.71s


[Vanilla Autoregressive (target model only)]
Generated text:
  The future of artificial intelligence is bright. December 3, 1999

Back to top

Cities of the future… design. with them, we shall show the truth about the speech of Hubbert. It is the rational part that is King. January 15, 2002

  Tokens generated:   50
  Target model 